# 0 · The coffee network, two ways: matrix vs. algebra

This notebook goes with the opening slides (*In a spreadsheet*, *As a matrix*, *As algebra*).

We solve **the same coffee network twice**, from **the same spreadsheet** (`data/coffee_network.xlsx`):

- **Part A, the matrix way.** Number every decision, build the cost vector `c`, the constraint matrix `A_ub` and the right-hand side `b_ub` by hand, and call `scipy.optimize.linprog`.
- **Part B, the algebra way.** Write the five lines of algebra in GAMSPy and let it build the matrix.

Then we check that both give the **same cheapest plan** and the **same shadow prices**.

Units: tonnes, and costs in $1,000 per tonne.

### Set up this notebook

Installs the packages this notebook needs into the running kernel, on Google Colab, Jupyter or VS Code. Already ran `pip install -r requirements.txt`? It just reports they are installed. It also checks the data files are present.

In [ ]:
%pip install -q gamspy pandas numpy scipy openpyxl plotly
# Optional: install your free academic license (otherwise the demo license is used)
# !gamspy install license <your-access-code>

# This notebook needs data files. If they are missing (e.g. on Colab), set REPO_RAW to the
# raw URL of the workshop repo, or upload them with Colab's Files pane (keep the data/ folder).
import os, urllib.request
REPO_RAW = ""   # e.g. "https://raw.githubusercontent.com/<user>/<repo>/main"
for path in ['data/coffee_network.xlsx']:
    if not os.path.exists(path):
        if REPO_RAW:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(f"{REPO_RAW}/{path}", path)
        else:
            print(f"Missing {path}: upload it, or set REPO_RAW above and re-run this cell.")

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.optimize import linprog
import gamspy as gp

## The spreadsheet

Open `data/coffee_network.xlsx` in Excel to see it. It has four sheets, laid out the way a person would type them in:

| Sheet | One row per | Columns |
|---|---|---|
| Farms | farm | harvest, cost to buy and ship to each roastery |
| Roasteries | roastery | roasting capacity, delivery cost to each city |
| Cafes | café city | demand for roasted coffee |
| Settings | setting | roasting yield (0.85) |

In [2]:
book = pd.read_excel("data/coffee_network.xlsx", sheet_name=None)
farms = book["Farms"].set_index("farm")
roasteries = book["Roasteries"].set_index("roastery")
cafes = book["Cafes"].set_index("city")
eta = float(book["Settings"].set_index("setting").loc["roasting_yield", "value"])

display(farms, roasteries, cafes)
print("roasting yield:", eta)

,name,harvest_t,cost_to_OAK,cost_to_BKN
farm,,,,
CO,Colombia,150,4.3,4.4
ET,Ethiopia,120,4.9,4.7
BR,Brazil,200,3.6,3.4


,name,capacity_t,deliver_to_CHI,deliver_to_AUS,deliver_to_DEN,deliver_to_ATL
roastery,,,,,,
OAK,Oakland,260,1.2,1.0,0.7,1.5
BKN,Brooklyn,180,0.7,1.2,1.1,0.6


,name,demand_t
city,,
CHI,Chicago,90
AUS,Austin,60
DEN,Denver,75
ATL,Atlanta,80


roasting yield: 0.85


---
# Part A · The matrix way

`linprog` knows nothing about farms or roasteries. It solves

$$\min_z \; c^\top z \quad \text{s.t.} \quad A_{ub}\, z \le b_{ub}, \; z \ge 0$$

for one long vector of numbers `z`. So our first job is **bookkeeping**: decide which position in `z` means which decision.

### Step A1 · Number the decisions (the columns)

In [3]:
F = farms.index.tolist()        # ['CO', 'ET', 'BR']
R = roasteries.index.tolist()   # ['OAK', 'BKN']
C = cafes.index.tolist()        # ['CHI', 'AUS', 'DEN', 'ATL']

columns = [("buy", f, r) for f in F for r in R] + [("ship", r, c) for r in R for c in C]
col = {name: j for j, name in enumerate(columns)}      # decision -> column number

pd.DataFrame({"column": range(len(columns)), "decision": [f"{k} {a}->{b}" for k, a, b in columns]})

,column,decision
0,0,buy CO->OAK
1,1,buy CO->BKN
2,2,buy ET->OAK
3,3,buy ET->BKN
4,4,buy BR->OAK
5,5,buy BR->BKN
6,6,ship OAK->CHI
7,7,ship OAK->AUS
8,8,ship OAK->DEN
9,9,ship OAK->ATL


**Quick question from the slides:** which column is Ethiopia → Brooklyn?

In [4]:
col[("buy", "ET", "BKN")]

3

### Step A2 · The cost vector `c`

Read each cost out of the spreadsheet and put it in the right position.

In [5]:
c = np.zeros(len(columns))
for f in F:
    for r in R:
        c[col[("buy", f, r)]] = farms.loc[f, f"cost_to_{r}"]
for r in R:
    for cc in C:
        c[col[("ship", r, cc)]] = roasteries.loc[r, f"deliver_to_{cc}"]
c

array([4.3, 4.4, 4.9, 4.7, 3.6, 3.4, 1.2, 1. , 0.7, 1.5, 0.7, 1.2, 1.1,
       0.6])

### Step A3 · The constraint matrix `A_ub` and right-hand side `b_ub`

One row per constraint, in a fixed order: harvest (3), capacity (2), balance (2), demand (4).

`linprog` only accepts **≤** rows, so the demand rows (which are **≥**) are multiplied by −1.

In [6]:
rows, A_ub, b_ub = [], [], []

def add_row(name, coefficients, rhs):
    row = np.zeros(len(columns))
    for decision, value in coefficients.items():
        row[col[decision]] = value
    rows.append(name); A_ub.append(row); b_ub.append(rhs)

for f in F:     # harvest: sum_r buy[f,r] <= H[f]
    add_row(f"harvest {f}", {("buy", f, r): 1 for r in R}, farms.loc[f, "harvest_t"])
for r in R:     # capacity: sum_f buy[f,r] <= K[r]
    add_row(f"capacity {r}", {("buy", f, r): 1 for f in F}, roasteries.loc[r, "capacity_t"])
for r in R:     # balance: sum_c ship[r,c] - eta * sum_f buy[f,r] <= 0
    add_row(f"balance {r}", {**{("ship", r, cc): 1 for cc in C}, **{("buy", f, r): -eta for f in F}}, 0)
for cc in C:    # demand: sum_r ship[r,c] >= D[c]   ->   -sum_r ship[r,c] <= -D[c]
    add_row(f"demand {cc}", {("ship", r, cc): -1 for r in R}, -cafes.loc[cc, "demand_t"])

A_ub, b_ub = np.array(A_ub), np.array(b_ub)
print(A_ub.shape, "->", len(rows), "constraints x", len(columns), "decisions")

(11, 14) -> 11 constraints x 14 decisions


Here is the matrix with labels. `linprog` only ever sees the numbers, never the labels.

In [7]:
labels = [f"{k} {a}->{b}" for k, a, b in columns]
fig = px.imshow(pd.DataFrame(A_ub, index=rows, columns=labels), text_auto=True, aspect="auto",
                color_continuous_scale=[[0, "#B35F00"], [0.5, "#FFFFFF"], [1, "#30383B"]],
                zmin=-1, zmax=1, title="A_ub: 11 constraints x 14 decisions")
fig.update_layout(width=1100, height=520, coloraxis_showscale=False)
fig.show()

### Check: the matrix on the slide

The *As a matrix* slide shows four rows of `A_ub` and the full `b_ub`. They should match what we just built.

In [8]:
slide_rows = {
    "harvest CO":  [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "harvest ET":  [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "harvest BR":  [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "balance OAK": [-.85, 0, -.85, 0, -.85, 0, 1, 1, 1, 1, 0, 0, 0, 0],
}
slide_c = [4.3, 4.4, 4.9, 4.7, 3.6, 3.4, 1.2, 1.0, 0.7, 1.5, 0.7, 1.2, 1.1, 0.6]
slide_b = [150, 120, 200, 260, 180, 0, 0, -90, -60, -75, -80]

for name, row in slide_rows.items():
    assert np.allclose(A_ub[rows.index(name)], row), name
assert np.allclose(c, slide_c) and np.allclose(b_ub, slide_b)
print("The slide's c, b_ub and A_ub rows match the matrix built from the spreadsheet.")

The slide's c, b_ub and A_ub rows match the matrix built from the spreadsheet.


### Step A4 · Solve, then translate the answer back

`linprog` returns 14 numbers. Turning them back into a plan takes the same bookkeeping in reverse.

In [9]:
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, None), method="highs")
print(res.message)
print(f"Total cost (matrix way): {res.fun:,.3f} ($1,000)")

z = pd.Series(res.x, index=pd.MultiIndex.from_tuples(columns, names=["kind", "from", "to"]))
buy_matrix = z["buy"].unstack().reindex(index=F, columns=R)
ship_matrix = z["ship"].unstack().reindex(index=R, columns=C)
display(buy_matrix.round(1), ship_matrix.round(1))

Optimization terminated successfully. (HiGHS Status 7: Optimal)
Total cost (matrix way): 1,604.235 ($1,000)


to,OAK,BKN
from,,
CO,150.0,0.0
ET,8.8,0.0
BR,20.0,180.0


to,CHI,AUS,DEN,ATL
from,,,,
OAK,17.0,60.0,75.0,0.0
BKN,73.0,0.0,0.0,80.0


---
# Part B · The algebra way

Same spreadsheet, but now we only reshape the data into long tables and write the algebra:

- harvest: $\sum_r x_{f,r} \le H_f$
- capacity: $\sum_f x_{f,r} \le K_r$
- balance: $\sum_c y_{r,c} \le \eta \sum_f x_{f,r}$
- demand: $\sum_r y_{r,c} \ge D_c$

No column numbers, no row order, no sign flips.

In [10]:
# reshape the wide spreadsheet columns into long (index, index, value) tables
price_long = (farms[[f"cost_to_{r}" for r in R]]
              .rename(columns=lambda s: s.replace("cost_to_", ""))
              .stack().reset_index())
deliver_long = (roasteries[[f"deliver_to_{cc}" for cc in C]]
                .rename(columns=lambda s: s.replace("deliver_to_", ""))
                .stack().reset_index())

m = gp.Container()
f = gp.Set(m, "f", records=F)
r = gp.Set(m, "r", records=R)
c_ = gp.Set(m, "c", records=C)

H = gp.Parameter(m, "H", domain=f, records=farms["harvest_t"].reset_index())
K = gp.Parameter(m, "K", domain=r, records=roasteries["capacity_t"].reset_index())
D = gp.Parameter(m, "D", domain=c_, records=cafes["demand_t"].reset_index())
p = gp.Parameter(m, "price", domain=[f, r], records=price_long)
d = gp.Parameter(m, "deliver", domain=[r, c_], records=deliver_long)

x = gp.Variable(m, "x", domain=[f, r], type="positive")
y = gp.Variable(m, "y", domain=[r, c_], type="positive")

harvest  = gp.Equation(m, "harvest",  domain=f)
capacity = gp.Equation(m, "capacity", domain=r)
balance  = gp.Equation(m, "balance",  domain=r)
demand   = gp.Equation(m, "demand",   domain=c_)

harvest[f]   = gp.Sum(r, x[f, r]) <= H[f]
capacity[r]  = gp.Sum(f, x[f, r]) <= K[r]
balance[r]   = gp.Sum(c_, y[r, c_]) <= eta * gp.Sum(f, x[f, r])
demand[c_]   = gp.Sum(r, y[r, c_]) >= D[c_]

coffee = gp.Model(m, "coffee", equations=m.getEquations(), problem="LP", sense="min",
                  objective=gp.Sum((f, r), p[f, r] * x[f, r]) + gp.Sum((r, c_), d[r, c_] * y[r, c_]))
coffee.solve()
print(coffee.status)
print(f"Total cost (algebra way): {coffee.objective_value:,.3f} ($1,000)")
display(x.pivot().reindex(index=F, columns=R).fillna(0).round(1),
        y.pivot().reindex(index=R, columns=C).fillna(0).round(1))

ModelStatus.OptimalGlobal
Total cost (algebra way): 1,604.235 ($1,000)


,OAK,BKN
CO,150.0,0.0
ET,8.8,0.0
BR,20.0,180.0


,CHI,AUS,DEN,ATL
OAK,17.0,60.0,75.0,0.0
BKN,73.0,0.0,0.0,80.0


> In Python we call the city set `c_`, because `c` is already the cost vector from Part A. In notebook 1 it's simply `c`.

---
# Compare the two

In [11]:
print(f"matrix way : {res.fun:,.6f}")
print(f"algebra way: {coffee.objective_value:,.6f}")
assert abs(res.fun - coffee.objective_value) < 1e-6
print("Same cheapest cost.")

buy_gamspy = x.pivot().reindex(index=F, columns=R).fillna(0)
ship_gamspy = y.pivot().reindex(index=R, columns=C).fillna(0)
print("Largest difference in the plans:",
      max(abs(buy_matrix - buy_gamspy).max().max(), abs(ship_matrix - ship_gamspy).max().max()))

matrix way : 1,604.235294
algebra way: 1,604.235294
Same cheapest cost.
Largest difference in the plans: 0.0


### The shadow prices agree too

Both solvers report a marginal for every constraint. The demand rows were flipped to ≤ for `linprog`, so their sign flips back.

In [12]:
lp_marginals = pd.Series(res.ineqlin.marginals, index=rows)

compare = pd.DataFrame({
    "matrix way (linprog)": pd.concat([
        lp_marginals[[f"harvest {i}" for i in F]].set_axis([f"harvest {i}" for i in F]),
        lp_marginals[[f"capacity {i}" for i in R]],
        -lp_marginals[[f"demand {i}" for i in C]],            # undo the sign flip
    ]),
    "algebra way (GAMSPy)": pd.concat([
        harvest.records.set_index("f")["marginal"].rename(lambda i: f"harvest {i}"),
        capacity.records.set_index("r")["marginal"].rename(lambda i: f"capacity {i}"),
        demand.records.set_index("c")["marginal"].rename(lambda i: f"demand {i}"),
    ]),
})
compare.round(4)

,matrix way (linprog),algebra way (GAMSPy)
harvest CO,-0.6000,-0.6000
harvest ET,-0.0000,0.0000
harvest BR,-1.3000,-1.3000
capacity OAK,-0.0000,0.0000
capacity BKN,-0.6250,-0.6250
demand CHI,6.9647,6.9647
demand AUS,6.7647,6.7647
demand DEN,6.4647,6.4647
demand ATL,6.8647,6.8647


---
## What this shows

| | Matrix way | Algebra way |
|---|---|---|
| You write | column numbers, row order, sign flips, unpacking | five lines that look like the math |
| Same answer? | yes | yes |
| Add a fourth farm | every column after it moves; the slide's hand-typed matrix is wrong | add a row to the spreadsheet; the model code is unchanged |

The code in Part A is general (it uses loops), and that's the point: to make the matrix way survive changes, you end up writing your own small modeling language. An AML is that, done properly.

## Your turn

1. **Add a fourth farm.** Add a row for Vietnam to the *Farms* sheet of the spreadsheet (VN, harvest 100 t, cost 3.9 to OAK and 4.2 to BKN), or add the line below right after the spreadsheet cell. Then use **Run All**. Do both parts still agree? Which column is Ethiopia → Brooklyn now?
2. **Re-run the slide check.** Why does it fail after you add Vietnam, even though both models are still correct?

In [13]:
# Your work here
# farms.loc["VN"] = ["Vietnam", 100, 3.9, 4.2]
